In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import numpy as np

# --- 1. Data Preparation (PyTorch Equivalent) ---

data = """ Jack and Jill went up the hill .\n To fetch a pail of water .\n Jack fell down and broke his crown .\n And Jill came tumbling after ."""
print("Data:", data, type(data))

data_splitted = data.split('\n') # returns a list of strings
print("Data_Splitted:", data_splitted, type(data_splitted))

# Custom Tokenizer to replicate Keras Tokenizer behavior
class PyTorchTokenizer:
    def __init__(self, filters='!"#$%&()*+,-/:;<=>?@[\]^_`{|}~'):
        self.word_index = {}
        self.index_word = {0: 'PAD'}
        self.filters = filters
        self.word_count = {}

    def fit_on_texts(self, texts):
        current_index = 1
        for text in texts:
            # Basic cleaning, remove filters, convert to lowercase
            cleaned_text = text.lower()
            for char in self.filters:
                cleaned_text = cleaned_text.replace(char, '')
            words = cleaned_text.split()
            for word in words:
                if word not in self.word_index:
                    self.word_index[word] = current_index
                    self.index_word[current_index] = word
                    current_index += 1
                self.word_count[word] = self.word_count.get(word, 0) + 1

    def texts_to_sequences(self, texts):
        sequences = []
        for text in texts:
            cleaned_text = text.lower()
            for char in self.filters:
                cleaned_text = cleaned_text.replace(char, '')
            words = cleaned_text.split()
            seq = [self.word_index[word] for word in words if word in self.word_index]
            sequences.append(seq)
        return sequences

# Instantiate and fit tokenizer
tokenizer_pt = PyTorchTokenizer()
tokenizer_pt.fit_on_texts(data_splitted)
print("Word Indices (PyTorch):", tokenizer_pt.word_index)

vocab_size_pt = len(tokenizer_pt.word_index) + 1
print("Vocab Size (PyTorch):", vocab_size_pt)

sequences_pt = tokenizer_pt.texts_to_sequences(data_splitted)
print("Sequences (PyTorch):", sequences_pt, type(sequences_pt), len(sequences_pt))

X_pt = []
y_pt = []

for seq in sequences_pt:
    X_pt.append(seq[:-1])
    y_pt.append(seq)

print("X_pt=", X_pt, "y_pt=", y_pt, type(X_pt), type(y_pt))

# Padding sequences
maxlen_pt = max([len(sequence) for sequence in X_pt])
print("Maxlen (PyTorch):", maxlen_pt)

# Keras pad_sequences defaults to 'pre' padding with 0. Maxlen is +1 for input X
def pad_sequences_pt(sequences, maxlen, padding='pre', value=0):
    padded_sequences = []
    for seq in sequences:
        if len(seq) >= maxlen:
            padded_sequences.append(seq[-maxlen:]) # Truncate if too long (Keras default is 'pre')
        else:
            pad_length = maxlen - len(seq)
            if padding == 'pre':
                padded_sequences.append([value] * pad_length + seq)
            elif padding == 'post':
                padded_sequences.append(seq + [value] * pad_length)
    return np.array(padded_sequences)

X_padded_pt = pad_sequences_pt(X_pt, maxlen=maxlen_pt + 1, padding='pre') # +1 to have 0 as the first input
y_padded_pt = pad_sequences_pt(y_pt, maxlen=maxlen_pt + 1, padding='pre')

print("X_padded_pt:", X_padded_pt, type(X_padded_pt), X_padded_pt.shape)
print("y_padded_pt:", y_padded_pt, type(y_padded_pt), y_padded_pt.shape)

# Convert to PyTorch tensors
X_tensor = torch.tensor(X_padded_pt, dtype=torch.long)

# y needs to be one-hot encoded for categorical_crossentropy, shape (batch, seq_len, vocab_size)
y_one_hot = F.one_hot(torch.tensor(y_padded_pt, dtype=torch.long), num_classes=vocab_size_pt).float()

print("X_tensor shape:", X_tensor.shape)
print("y_one_hot shape:", y_one_hot.shape)


# --- 2. Model Definition (PyTorch Equivalent) ---

class PyTorchRNN(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_size):
        super(PyTorchRNN, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.rnn = nn.RNN(embedding_dim, hidden_size, batch_first=True) # batch_first=True for (batch, seq, feature)
        self.dense = nn.Linear(hidden_size, vocab_size)

    def forward(self, x):
        embedded = self.embedding(x)
        output, _ = self.rnn(embedded) # _ is the hidden state, not used here for simple RNN
        # Apply dense layer to each time step's output
        output = self.dense(output)
        return output


embedding_dim = 10
hidden_size = 50
model_pt = PyTorchRNN(vocab_size_pt, embedding_dim, hidden_size)
print(model_pt)


# --- 3. Training Loop (PyTorch Equivalent) ---

optimizer_pt = optim.RMSprop(model_pt.parameters())
# CrossEntropyLoss expects (N, C, ...) for input and (N, ...) for target
# Our output is (batch, seq_len, vocab_size), target is (batch, seq_len, vocab_size) one-hot
# We need to reshape and then apply F.log_softmax and NLLLoss, or flatten and use CrossEntropyLoss

# For simplicity, we will flatten and use CrossEntropyLoss which combines LogSoftmax and NLLLoss
# Input to CrossEntropyLoss should be (N, C) and target (N) where N is total number of elements

def train_pytorch_model(model, X_train, y_train_one_hot, epochs, optimizer):
    model.train() # Set the model to training mode
    criterion = nn.CrossEntropyLoss() # This implicitly applies LogSoftmax and then NLLLoss

    # Reshape y_train_one_hot to (batch_size * seq_len, vocab_size)
    # And get the target indices from the one-hot encoding for CrossEntropyLoss
    y_target_indices = torch.argmax(y_train_one_hot, dim=-1) # (batch_size, seq_len)

    for epoch in range(epochs):
        optimizer.zero_grad()

        # Forward pass
        outputs = model(X_train) # (batch_size, seq_len, vocab_size)

        # Reshape outputs for CrossEntropyLoss: (batch_size * seq_len, vocab_size)
        outputs_flat = outputs.view(-1, vocab_size_pt)
        # Reshape target indices: (batch_size * seq_len)
        targets_flat = y_target_indices.view(-1)

        loss = criterion(outputs_flat, targets_flat)
        loss.backward() # Backward pass
        optimizer.step() # Optimize

        if (epoch + 1) % 50 == 0:
            print(f'Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}')

epochs_pt = 200
train_pytorch_model(model_pt, X_tensor, y_one_hot, epochs_pt, optimizer_pt)

print("\n--- PyTorch Model Training Complete ---")


# --- 4. Probability Calculation Function (PyTorch Equivalent) ---

def prob_of_input_sentence_pt(model, tokenizer, sentence, vocab_size):
    model.eval() # Set the model to evaluation mode
    with torch.no_grad(): # Disable gradient calculations
        print("Input Sentence:", sentence)

        # Tokenize the sentence using the custom tokenizer
        encoded_list = tokenizer.texts_to_sequences([sentence])[0]

        # Keras `pad_sequences` behavior: inserts 0 at the beginning if input is shorter than maxlen.
        # Our X was padded with +1, so we need to match that.
        # The original Keras code inserted a 0 at the beginning manually, then padded.
        # We need to consider how the original maxlen was defined (maxlen_pt is for X_pt, which is already `seq[:-1]`).
        # The `X_tensor` was padded to `maxlen_pt + 1`.
        # So, the input to the model should be of `maxlen_pt + 1`.

        # First, ensure the '0' prefix for empty/short sequences as in Keras example
        # Original Keras code: encoded.insert(0,0) - this inserts 0 irrespective of actual length, then pad_sequences
        # So if `sentence` tokenizes to `[1, 2, 3]`, encoded becomes `[0, 1, 2, 3]` and then padded.

        input_seq_for_prob = [0] + encoded_list # Prepend 0 as in original Keras

        # Pad to the length the model expects, which is maxlen_pt + 1
        # Use our custom pad_sequences_pt with 'pre' padding
        padded_input = pad_sequences_pt([input_seq_for_prob], maxlen=maxlen_pt + 1, padding='pre')[0]

        encoded_tensor = torch.tensor(padded_input, dtype=torch.long).unsqueeze(0) # Add batch dimension
        print("Encoded (PyTorch):", encoded_tensor, encoded_tensor.shape)

        # Get model predictions (logits)
        logits = model(encoded_tensor) # (1, seq_len, vocab_size)

        # Apply softmax to get probabilities for each token at each step
        prob = F.softmax(logits, dim=-1) # (1, seq_len, vocab_size)
        print("Prob (PyTorch):", prob, prob.shape)

        probability = 1.0

        # Iterate through the sequence to calculate the joint probability
        # The Keras code calculated P(word_i | word_0...word_{i-1})
        # and multiplied these probabilities. This corresponds to the probability
        # of the actual next word in the sequence, given the previous ones.

        # The original Keras code iterates from i=0 to prob.shape[1]-1 (length-1)
        # and uses encoded[0, i+1] as the target word.

        # So for an input sequence `s_0, s_1, ..., s_N`, the model predicts `p_0, p_1, ..., p_N`
        # `p_0` is prob of `s_1` given `s_0`
        # `p_1` is prob of `s_2` given `s_0, s_1` and so on.
        # The Keras `encoded[0, i+1]` is the actual next word at position `i+1`.

        for i in range(prob.shape[1] - 1):
            # prob[0, i, :] gives the probability distribution for the next word at step i
            # encoded_tensor[0, i+1] is the actual next word index in the input sequence

            # Ensure the index is within bounds of vocab_size
            target_word_index = int(encoded_tensor[0, i+1].item())
            if target_word_index >= vocab_size:
                # This case might happen if token not in vocab, or it's a padding zero.
                # If it's a padding zero, its probability should be 1 if it's the target
                # Or we skip it if it's not a real word. Keras handles padding '0' gracefully.
                # We will assume 0 (PAD) has a probability of 1 if it's the target (for padding effect)
                # But if a word from the sentence itself is not in vocab, its prob is 0.
                # For this specific Keras example, all words are in vocab.
                # If target_word_index is 0 (PAD), we treat it as 1 for simplicity of padding effect.
                if target_word_index == 0:
                    word_prob = 1.0 # Or maybe skip. Original Keras would try to predict prob of 0.
                                    # For padding, it usually means we're done with the sentence.
                                    # The original Keras prob was calculated for 0 as well.
                else:
                    word_prob = 0.0 # Word not in vocab
            else:
                word_prob = prob[0, i, target_word_index].item()
            probability *= word_prob

        print("Probability of Sentence ", "\"" + sentence + "\"", "is:", probability)


print("-------------------Probability of Input Sentence (PyTorch)------------------------------")
prob_of_input_sentence_pt(model_pt, tokenizer_pt, "Jack and Jill Went", vocab_size_pt)
prob_of_input_sentence_pt(model_pt, tokenizer_pt, "and Jill Went up", vocab_size_pt)
prob_of_input_sentence_pt(model_pt, tokenizer_pt, "went up the hill", vocab_size_pt)
prob_of_input_sentence_pt(model_pt, tokenizer_pt, "jack and Jill Went up the hill .", vocab_size_pt)
prob_of_input_sentence_pt(model_pt, tokenizer_pt, "to fetch a pail", vocab_size_pt)
prob_of_input_sentence_pt(model_pt, tokenizer_pt, "fetch a pail", vocab_size_pt)
prob_of_input_sentence_pt(model_pt, tokenizer_pt, "to fetch a pail of water", vocab_size_pt)
prob_of_input_sentence_pt(model_pt, tokenizer_pt, "jack fell down and", vocab_size_pt)
prob_of_input_sentence_pt(model_pt, tokenizer_pt, "jack fell down and broke", vocab_size_pt)
prob_of_input_sentence_pt(model_pt, tokenizer_pt, "fell down and broke", vocab_size_pt)
prob_of_input_sentence_pt(model_pt, tokenizer_pt, "jack fell down and broke his crown", vocab_size_pt)
prob_of_input_sentence_pt(model_pt, tokenizer_pt, "and jill came tumbling", vocab_size_pt)
prob_of_input_sentence_pt(model_pt, tokenizer_pt, "g after", vocab_size_pt)
prob_of_input_sentence_pt(model_pt, tokenizer_pt, "and jill came tumbling after .", vocab_size_pt)
prob_of_input_sentence_pt(model_pt, tokenizer_pt, "and jill came walkings after .", vocab_size_pt)
print("-------------------------------------------------------------------------------")

<>:17: SyntaxWarning: invalid escape sequence '\]'
<>:17: SyntaxWarning: invalid escape sequence '\]'
/tmp/ipython-input-1083552682.py:17: SyntaxWarning: invalid escape sequence '\]'
  def __init__(self, filters='!"#$%&()*+,-/:;<=>?@[\]^_`{|}~'):


Data:  Jack and Jill went up the hill .
 To fetch a pail of water .
 Jack fell down and broke his crown .
 And Jill came tumbling after . <class 'str'>
Data_Splitted: [' Jack and Jill went up the hill .', ' To fetch a pail of water .', ' Jack fell down and broke his crown .', ' And Jill came tumbling after .'] <class 'list'>
Word Indices (PyTorch): {'jack': 1, 'and': 2, 'jill': 3, 'went': 4, 'up': 5, 'the': 6, 'hill': 7, '.': 8, 'to': 9, 'fetch': 10, 'a': 11, 'pail': 12, 'of': 13, 'water': 14, 'fell': 15, 'down': 16, 'broke': 17, 'his': 18, 'crown': 19, 'came': 20, 'tumbling': 21, 'after': 22}
Vocab Size (PyTorch): 23
Sequences (PyTorch): [[1, 2, 3, 4, 5, 6, 7, 8], [9, 10, 11, 12, 13, 14, 8], [1, 15, 16, 2, 17, 18, 19, 8], [2, 3, 20, 21, 22, 8]] <class 'list'> 4
X_pt= [[1, 2, 3, 4, 5, 6, 7], [9, 10, 11, 12, 13, 14], [1, 15, 16, 2, 17, 18, 19], [2, 3, 20, 21, 22]] y_pt= [[1, 2, 3, 4, 5, 6, 7, 8], [9, 10, 11, 12, 13, 14, 8], [1, 15, 16, 2, 17, 18, 19, 8], [2, 3, 20, 21, 22, 8]] <class 'l

# Implementing Next Word Prediction

In [ ]:
def predict_next_word(model, tokenizer, text_phrase, N=5):
    model.eval() # Set the model to evaluation mode
    with torch.no_grad(): # Disable gradient calculations
        # Tokenize the text_phrase
        encoded_list = tokenizer.texts_to_sequences([text_phrase])[0]

        # Prepend 0 and pad the sequence to maxlen_pt + 1
        input_seq_for_pred = [0] + encoded_list
        padded_input = pad_sequences_pt([input_seq_for_pred], maxlen=maxlen_pt + 1, padding='pre')[0]

        # Convert to PyTorch tensor and add batch dimension
        input_tensor = torch.tensor(padded_input, dtype=torch.long).unsqueeze(0)

        # Get model predictions (logits)
        logits = model(input_tensor)

        # Apply softmax to get probabilities
        probabilities = F.softmax(logits, dim=-1)

        # Extract probabilities for the last token (prediction for the next word)
        # The prediction for the next word is based on the entire input_tensor up to the last actual word.
        # So we look at the probabilities at the last position of the sequence output.
        last_word_prob_dist = probabilities[0, -1, :]

        # Get top N probabilities and their indices
        top_n_probs, top_n_indices = torch.topk(last_word_prob_dist, N)

        # Convert indices back to words
        predictions = []
        for i in range(N):
            word_index = top_n_indices[i].item()
            word = tokenizer.index_word.get(word_index, '<UNK>') # Handle potential unknown words
            prob = top_n_probs[i].item()
            predictions.append((word, prob))

        return predictions

print("Next word prediction function defined.")

Next word prediction function defined.


In [ ]:
print("-------------------Next Word Prediction (PyTorch)------------------------------")

phrases_to_predict = [
    "Jack and Jill",
    "went up the",
    "to fetch a",
    "Jack fell down and broke his",
    "Jill came tumbling"
]

for phrase in phrases_to_predict:
    print(f"\nInput phrase: '{phrase}'")
    predictions = predict_next_word(model_pt, tokenizer_pt, phrase, N=5)
    print("Top 5 next word predictions:")
    for word, prob in predictions:
        print(f"  - {word}: {prob:.4f}")

print("-------------------------------------------------------------------------------")

-------------------Next Word Prediction (PyTorch)------------------------------

Input phrase: 'Jack and Jill'
Top 5 next word predictions:
  - went: 0.9944
  - came: 0.0031
  - and: 0.0006
  - his: 0.0006
  - a: 0.0002

Input phrase: 'went up the'
Top 5 next word predictions:
  - hill: 0.9968
  - after: 0.0014
  - up: 0.0004
  - down: 0.0003
  - water: 0.0002

Input phrase: 'to fetch a'
Top 5 next word predictions:
  - pail: 0.9993
  - .: 0.0002
  - up: 0.0001
  - crown: 0.0001
  - broke: 0.0001

Input phrase: 'Jack fell down and broke his'
Top 5 next word predictions:
  - crown: 0.9995
  - fetch: 0.0001
  - pail: 0.0001
  - broke: 0.0001
  - water: 0.0000

Input phrase: 'Jill came tumbling'
Top 5 next word predictions:
  - after: 0.9899
  - hill: 0.0069
  - up: 0.0008
  - broke: 0.0006
  - went: 0.0004
-------------------------------------------------------------------------------
